<a href="https://colab.research.google.com/github/juii18/ArogyaMitra/blob/main/Sentiment%20Classification%20of%20Movies%20using%20IMDb%20Ratings%20and%20NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Importing Libraries

In [13]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

2. Load Dataset

In [14]:
from google.colab import drive
drive.mount('/content/drive')

# Change path according to your Drive
file_path = "/content/drive/MyDrive/IMDb_All_Genres_etf_clean1.csv"

df = pd.read_csv(file_path)

print(df.head())
print(df.columns)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
                                     Movie_Title  Year           Director  \
0                                        Kantara  2022      Rishab Shetty   
1                                The Dark Knight  2008  Christopher Nolan   
2  The Lord of the Rings: The Return of the King  2003      Peter Jackson   
3                                      Inception  2010  Christopher Nolan   
4          The Lord of the Rings: The Two Towers  2002      Peter Jackson   

                                              Actors  Rating  Runtime(Mins)  \
0  Rishab Shetty, Sapthami Gowda, Kishore Kumar G...     9.3            148   
1  Christian Bale, Heath Ledger, Aaron Eckhart, M...     9.0            152   
2  Elijah Wood, Viggo Mortensen, Ian McKellen, Or...     9.0            201   
3  Leonardo DiCaprio, Joseph Gordon-Levitt, Ellio...     8.8            148   
4  Elijah Woo

3. Create Sentiment from Rating

In [15]:
def get_sentiment(rating):
    if rating >= 7:
        return "Positive"
    elif rating >= 5:
        return "Neutral"
    else:
        return "Negative"

df['sentiment'] = df['Rating'].apply(get_sentiment)

4. Create Text (since no review column exists)

In [16]:
df['review'] = (
    df['Movie_Title'].astype(str) + " directed by " +
    df['Director'].astype(str) + " has rating " +
    df['Rating'].astype(str)
)

5. Clean Text

In [24]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df['clean_review'] = df['review'].apply(clean_text)

6. Convert Text → Numbers (TF-IDF)

In [18]:
vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df['clean_review']).toarray()
y = df['sentiment']

7. Train-Test Split

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

8. Train Model

In [20]:
model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

LogisticRegression(max_iter=200)

9. Evaluate Model

In [21]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.628032345013477

Classification Report:

              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00        37
     Neutral       0.64      0.73      0.68       596
    Positive       0.61      0.55      0.58       480

    accuracy                           0.63      1113
   macro avg       0.42      0.43      0.42      1113
weighted avg       0.61      0.63      0.61      1113



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


10. Testing Model

In [23]:
def predict_sentiment(text):
    text = clean_text(text)
    vector = vectorizer.transform([text]).toarray()
    return model.predict(vector)[0]

# Test input
user_input = input("Enter a movie review-like text: ")
print("Predicted Sentiment:", predict_sentiment(user_input))

Enter a movie review-like text: kung fu
Predicted Sentiment: Positive
